# 06 - Verification: self-tests, traces, compatibility, overhead, gates

**Purpose:** Run ML-KEM / ML-DSA self-tests, before/after loopback TLS traces and the compatibility matrix with OpenSSL 3.5; show saved overhead numbers and gate results.

**Inputs:** Git-bundled OpenSSL 3.5.9 (full path), throw-away test keys in a temp dir

**Outputs:** tables of results; gate states per step

**Status labels:** VERIFIED, TEST SCENARIO (loopback, test keys). Self-tests are round-trip tests, NOT NIST KATs. Overhead is loopback only.

TEST/demo context: the demo bank in `demo_bank_app/` is SYNTHETIC.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
DERIVED.mkdir(parents=True, exist_ok=True)
# This notebook reads no credentials. If a future cell needs one, read it from an environment variable at runtime
# and never display it.

In [ ]:
import pandas as pd
from cryptoshift.gates import openssl_lab as lab
print(lab.capabilities())
with lab.lab_dir() as d:
    st = [lab.mlkem_roundtrip_openssl(d), lab.mlkem_roundtrip_cryptography(), lab.mldsa_sign_verify_openssl(d), lab.mldsa_sign_verify_cryptography()]
    tr = lab.traces(d)
    cm = lab.compatibility_matrix(d)
pd.DataFrame(st)[['implementation','algorithm','status','note']]

In [ ]:
pd.DataFrame([{'trace': k, **{f: v.get(f) for f in ('handshake_ok','protocol','group','cipher','bytes_read','bytes_written','peer_signature_type','command')}} for k, v in tr.items() if isinstance(v, dict)])

In [ ]:
pd.DataFrame(cm)[['case','category','expected','observed','pass']]

In [ ]:
ev = json.loads((DERIVED / 'verification_evidence.json').read_text()) if (DERIVED / 'verification_evidence.json').is_file() else {}
ev.get('overhead', {}).get('results', 'run scripts/member3_phase4.py first')

In [ ]:
from cryptoshift.gates.verification_gates import evaluate_plan
from cryptoshift.planning.planner import build_problem
steps = evaluate_plan(build_problem()['services'], ev)
pd.DataFrame([{'step': s['step_id'], 'state': s['state'], 'blocking': s['blocking_gates']} for s in steps])

READY means the generic loopback evidence passed; it does not exercise each service's own stack (KMS wrapping, JWT libraries, partner peers). Thresholds are assumptions (docs/member3/assumptions.md).